# Reproducing Mountoufaris et al. (2024) from dandiset 001131: activity, selectivity, decoding and persistence

Paper. Mountoufaris G, Nair A, Yang B, Kim D-W, Vinograd A, Kim S, Linderman SW, Anderson DJ (2024).
A line attractor encoding a persistent internal state requires neuropeptide signaling. *Cell* 187,
5998-6015. [doi:10.1016/j.cell.2024.08.015](https://doi.org/10.1016/j.cell.2024.08.015). The claims
file (`claims/001131/10.1016-j.cell.2024.08.015.yaml`) was written from the accepted author manuscript
in PubMed Central (PMC11490375), with the supplementary figures; this attempt read the same text.

Dandiset. [DANDI:001131](https://dandiarchive.org/dandiset/001131/0.260601.1203), version
0.260601.1203. The paper cites the dandiset without a version (Key resources table, "DANDI:001131").
Two versions are published. 0.240826.1647 holds two files, one control and one experimental mouse;
0.260601.1203, the most recent, holds ten, and its files for mice 510 and 500 hold the same data as
the two earlier files (checked below). The most recent version is pinned because it holds all the
data of the earlier one and more, and so matches the claims file's description better.

What the dandiset holds. One file per male resident: 4 control (scrambled gRNA) and 6 experimental
(Oxtr/Avpr1a gRNAs) mice, each with a z-scored trace per unit at 10 Hz and the annotated sniffing,
attack and dominance mounting towards a male intruder (two files only sniffing). Two files hold the
authors' rSLDS weights of each unit on "latent factor 1". There are no female-intruder sessions, no
mark of the pre-intruder baseline, the intruder's entry or removal, and no raw dF/F. The behaviour
and photometry cohorts (Figures 2, 3), the slice imaging (Figures 1F, 2E, S7C-L) and the outside
RNA-seq atlas (Figure 1E) are not in it. The atlas, of Kim et al. 2019, is read from Mendeley Data
(doi:10.17632/ypx3sw2f7c.1) for the fig1e test.

This notebook tests the claims that need no rSLDS fit. The rSLDS claims are in `rslds_seeds_0_1.ipynb`
and `rslds_seeds_2_3.ipynb` (the four seeds the claims file asks for do not fit in one notebook's
time limit), and `rslds_checks.ipynb` holds the sigma-normalized and held-out checks of seed 0.

| Claim | Here |
|---|---|
| methods-unit-counts | tested |
| figs4a-unilateral-no-behavior-deficit | tested (attack probability assumed) |
| fig4ef-male-activity-slightly-lower | tested (baseline assumed) |
| figs4e-active-units-unchanged | tested (baseline assumed) |
| figs5c-peth-activity | tested on its male part, which fails; the female part is not in the dandiset |
| fig5bd-behavior-selective-units-reduced | tested (baseline assumed) |
| figs5e-behavior-decoding-intact | tested |
| fig7c-achw-all-units | tested |
| fig5ac-behavior-activity-male-lower | blocked, data missing (female part); male part shown |
| fig5e-cp-tuning-shift | blocked, data missing (female sniffing); male part shown |
| fig4g-male-selective-reduced | blocked, data missing (female part); male-activated fractions shown |
| fig4ef-female-activity-unchanged, fig4g-female-selective-and-mixed-increased, fig4d-sex-decoding-intact, fig4c-pls-sex-separation, figs7b-achw-female-slight | blocked, data missing (no female-intruder data) |
| fig1e-oxtr-avpr1a-coexpression | tested on the outside atlas (cell set and threshold assumed) |
| fig1f, fig2e, fig2g, fig2 (sniff and mating), fig3c, fig3d, fig3fh, figs3ef, figs7cd, figs7k, figs7l | blocked (data not in the dandiset) |
| fig6-rslds-fit-quality, fig6a, fig6c, fig6f, fig6g, fig7b | tested in the rSLDS notebooks (checks in `rslds_checks.ipynb`) |
| fig6e-integration-dimension-persistence | blocked, metadata insufficient (no intruder removal times); variant in the rSLDS notebooks |
| figs6f-mating-attractor-intact | blocked, data missing (female) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import vmh

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
mice = vmh.load_all()
G = {m["mouse"]: m["group"] for m in mice}
M = {m["mouse"]: m for m in mice}
print(f"loaded {len(mice)} files from dandiset {vmh.DANDISET_ID} version {vmh.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

In [ ]:
info = pd.DataFrame([{
    "mouse": m["mouse"], "group": m["group"], "path": m["path"], "frames": m["Y"].shape[0],
    "units": m["Y"].shape[1], "rate_hz": m["rate"], "duration_s": m["Y"].shape[0] / m["rate"],
    "first_behaviour_s": vmh.first_behaviour_s(m),
    **{f"{b}_bouts": len(m["bouts"].get(b, [])) for b in vmh.BEHAVIOURS},
    "rslds_weights": m["rslds_weights"] is not None, "session_start": m["session_start"][:10],
    "trace_unit": m["trace_unit"]} for m in mice])
print(info.round(1).to_string(index=False))
print("\nMice and units per group:")
print(info.groupby("group").agg(mice=("mouse", "count"), units=("units", "sum"),
                                frames=("frames", "sum")).to_string())
print("\nInterval series names per file:", sorted({k for m in mice for k in m["series"]}))
print("Sampling rate of every trace:", sorted({m["rate"] for m in mice}), "Hz; starting time",
      sorted({m["starting_time"] for m in mice}))

The group is read from the subject id (`ctrl...` and `mut...`; the keywords of eight files also say
"control" or "mutant"). The dandiset holds 4 control and 6 experimental mice, where the STAR Methods
give 5 and 7. Eight files name their behaviours `Ch1_sniffing`, `Ch1_attack` and
`Ch1_dominance_mounting`; the files of mice 510 and 500 hold only `Ch1: sniffing`, although the
paper's exemplar mice attack (Figure 6B, D). Every session but one has the same placeholder start
date, 2023-02-01.

In [ ]:
rows = []
for m in mice:
    Y = m["Y"]
    rows.append({"mouse": m["mouse"], "unit_mean_absmax": np.abs(Y.mean(0)).max(),
                 "unit_sd_min": Y.std(0, ddof=1).min(), "unit_sd_max": Y.std(0, ddof=1).max(),
                 "min": Y.min(), "max": Y.max(), "nan": int(np.isnan(Y).sum()),
                 "described_as": m["trace_description"][:60], "comments": m["trace_comments"][:60]})
print(pd.DataFrame(rows).round(4).to_string(index=False))

Every unit of nine files has mean 0 and SD 1 over its file: the traces are z-scored over the whole
file, as eight files say. In mouse 500 the SDs are below 1 (from the table), so its trace was
z-scored over a longer recording and then cut. The files of mice 510 and 500 describe the same kind
of values as "DF/F" from "GCaMP6m". The values are z-scores (table), not dF/F, and the paper says
the imaged mice carried "an AAV vector that encodes both the gRNAs and a Cre-dependent GCaMP8s"
(Results, Single cell CRISPRoscopy imaging; Figure 3A), so the indicator named in these two files
disagrees with the paper. Units of sigma above a baseline are computed from the
z-scored trace: z is an affine map of each unit's dF/F, so (z - baseline mean) / baseline SD is the
same as the Methods' normalization of dF/F.

In [ ]:
# Behaviour intervals: onsets and offsets alternate, lie inside the recording, and do not overlap.
rows = []
for m in mice:
    T = len(m["Y"]) / m["rate"]
    for beh, b in m["bouts"].items():
        d, t = m["series"][m["bout_series"][beh]]
        rows.append({"mouse": m["mouse"], "behaviour": beh, "bouts": len(b),
                     "alternating": bool(np.all(d[::2] == 1) and np.all(d[1::2] == -1)),
                     "inside_recording": bool(b.min() >= 0 and b.max() <= T),
                     "overlapping": int((b[1:, 0] < b[:-1, 1]).sum()),
                     "dur_min_s": (b[:, 1] - b[:, 0]).min(), "dur_median_s": np.median(b[:, 1] - b[:, 0]),
                     "dur_max_s": (b[:, 1] - b[:, 0]).max(), "first_s": b[0, 0], "last_s": b[-1, 1],
                     "recording_s": T})
print(pd.DataFrame(rows).round(1).to_string(index=False))
both = {}
for m in mice:
    if "attack" in m["bouts"]:
        n = len(m["Y"])
        both[m["mouse"]] = int((vmh.frames(m["bouts"]["sniff"], n) & vmh.frames(m["bouts"]["attack"], n)).sum())
print("\nFrames annotated both sniff and attack, per mouse:", both)

Mouse 511 has only two sniffing bouts, of 22 and 40 s, against tens of short bouts in the other
mice. Its sniffing annotation is kept as it is.

In [ ]:
# What each file says it holds, and whether anything points to a female intruder.
rows = []
for m in mice:
    n = len(m["Y"])
    last = max(b[-1, 1] for b in m["bouts"].values())
    rows.append({"mouse": m["mouse"], "session_description": m["session_description"],
                 "keywords": ",".join(m["keywords"]), "series": ",".join(sorted(m["series"])),
                 "last_annotation_s": round(float(last), 1), "recording_s": n / m["rate"],
                 "female_or_ch2_words": any(w in (m["session_description"] + m["behavior_description"] + " ".join(m["series"])).lower()
                                            for w in ("female", "ch2", "mating", "mount female"))})
print(pd.DataFrame(rows).to_string(index=False))
print("\nbehaviour module descriptions:", sorted({m["behavior_description"] for m in mice}))

Every file is described as a "resident-intruder assay"; the two exemplar files (mice 510 and 500)
add "with rSLDS integration weights". No description, keyword or series name mentions a female or a
second channel (`Ch2`), and every series is named `Ch1`. That the files hold only the male-intruder
part of each session is therefore consistent with the files, not shown: nothing in them marks a
female interaction, and the Methods' sessions had the male intruder first (Behavior annotations).

In [ ]:
# Duplicates. (1) The two files of the earlier version against the ten of the pinned one.
old = vmh.load_all(vmh.EARLIER_VERSION)
for o in old:
    same = [m["mouse"] for m in mice if m["Y"].shape == o["Y"].shape and np.array_equal(m["Y"], o["Y"])]
    sb = [m["mouse"] for m in mice if same and m["mouse"] in same and
          np.array_equal(m["bouts"]["sniff"], o["bouts"]["sniff"]) and
          np.array_equal(m["rslds_weights"], o["rslds_weights"])]
    print(f"{vmh.EARLIER_VERSION} {o['path']} (subject {o['mouse']}): identical trace in {same}; "
          f"identical sniffing bouts and rSLDS weights in {sb}")
# (2) Any unit trace repeated within or across the ten files, and the largest correlation between
# population means of two mice over their common length.
cols = {}
for m in mice:
    for j in range(m["Y"].shape[1]):
        cols.setdefault(np.round(m["Y"][:, j], 6).tobytes(), []).append((m["mouse"], j))
print("unit traces that occur more than once:", sum(len(v) > 1 for v in cols.values()))
best = (0, None)
names = [m["mouse"] for m in mice]
for i in range(len(mice)):
    for j in range(i + 1, len(mice)):
        n = min(len(mice[i]["Y"]), len(mice[j]["Y"]))
        r = np.corrcoef(mice[i]["Y"][:n].mean(1), mice[j]["Y"][:n].mean(1))[0, 1]
        if abs(r) > abs(best[0]):
            best = (r, (names[i], names[j]))
print("largest correlation of population means between two mice: %.3f %s" % best)
del old, cols

The two files of version 0.240826.1647 ("ControlMouse" and "MutantMouse") are mice 510 and 500 of
the pinned version, with identical traces, sniffing bouts and rSLDS weights, so the earlier version
adds nothing and each recording is counted once. No unit trace occurs twice, and no two mice share a
recording (largest correlation of population means printed above).

### The clock of traces and annotations

The traces and the behaviour annotations are given on one clock (traces start at 0 s at 10 Hz,
interval timestamps in seconds). To check that they line up, a decoder of "attack or sniff" against
all other frames is trained on the first half of each session and tested on the second, with the
labels shifted in time. If the clocks agree, the decoder does best at a shift near zero.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

shifts = [-30, -10, -5, -2, 0, 2, 5, 10, 30]
rows = []
for m in mice:
    Y, n = m["Y"], len(m["Y"])
    lab = np.zeros(n, bool)
    for b in m["bouts"].values():
        lab |= vmh.frames(b, n)
    # frames from the first to the last annotated behaviour, in four blocks: train on 1 and 3, test on 2 and 4
    a = int(min(b[0, 0] for b in m["bouts"].values()) * m["rate"])
    z = int(max(b[-1, 1] for b in m["bouts"].values()) * m["rate"])
    block = np.arange(z - a) * 4 // (z - a)
    train = np.isin(block, [0, 2])
    r = {"mouse": m["mouse"]}
    for s in shifts:
        y = np.roll(lab, int(s * m["rate"]))[a:z]  # label of frame t is the annotation at t - s
        if y[train].mean() in (0, 1) or y[~train].mean() in (0, 1):
            r[s] = np.nan
            continue
        clf = LogisticRegression(C=0.1, max_iter=2000).fit(Y[a:z][train], y[train])
        r[s] = roc_auc_score(y[~train], clf.decision_function(Y[a:z][~train]))
    rows.append(r)
clock = pd.DataFrame(rows).set_index("mouse")
print("Test AUC of the behaviour decoder by label shift (s), within the interaction; positive shift = labels delayed:")
print(clock.round(3).to_string())
print("\nshift of the best AUC per mouse:", clock.idxmax(axis=1).to_dict())
print("best AUC per mouse:", clock.max(axis=1).round(3).to_dict())

Within the interaction (first to last annotated behaviour), a decoder of "any annotated behaviour"
against the frames between bouts is trained on two interleaved quarters and tested on the other
two. In mice 482, 508, 511, 440, 441, 488 and 501 it does best at a shift of 2 to 5 s (activity
following the annotation, as a calcium signal would) and worse at 30 s in either direction, which is
consistent with traces and annotations sharing a clock. In mice 497, 510 and 500 the AUC changes
little with the shift or peaks far from zero (table), so their alignment could not be established;
they are kept, and this is said where it bears on a result.

### The pre-intruder baseline

The claims in units of sigma need the pre-intruder baseline, which the files do not mark (the
Methods say a 5-min baseline alone starts every session). The frames before the first annotated
behaviour are taken as the pre-intruder period (an assumption; see `vmh.py`). The population mean
below shows whether activity changes at the first annotated behaviour.

In [ ]:
rows = []
for m in mice:
    pm, i0 = m["Y"].mean(1), int(round(vmh.first_behaviour_s(m) * m["rate"]))
    seg = lambda a, b: pm[max(0, i0 + int(a * 10)):max(0, i0 + int(b * 10))].mean() if i0 + int(b * 10) > 0 else np.nan
    rows.append({"mouse": m["mouse"], "pre_s": i0 / 10, "baseline_used": i0 >= 70,
                 "z[-60,-10]": seg(-60, -10), "z[-10,0]": seg(-10, 0), "z[0,30]": seg(0, 30),
                 "z[30,120]": seg(30, 120)})
print("Population mean z-score in windows relative to the first annotated behaviour (s):")
print(pd.DataFrame(rows).round(2).to_string(index=False))

Mean activity rises after the first annotated behaviour in every mouse with a baseline (table). In
mouse 482 it is already higher in the 10 s before it, which suggests the intruder entered a few
seconds before the first annotated sniff; the variant baseline ending 10 s before the first
behaviour (`gap10`) is shown alongside the two readings for that reason. Mouse 488 has only 6.2 s
before its first annotation, less than the Methods' "7sec or more", and is left out of every
sigma-based measure. Only mice 508 and 511 have more than 5 min before the first annotation.

## Claim methods-unit-counts

> CRISPRoscopy recorded 585 units from 5 control mice and 546 units from 7 experimental mice.

In [ ]:
counts = info.groupby("group").agg(mice=("mouse", "count"), units=("units", "sum"))
counts["paper_mice"] = [5, 7]
counts["paper_units"] = [585, 546]
print(counts.to_string())
print("\nunits per mouse:", info.set_index("mouse")["units"].to_dict())

**Result: not reproduced.** The dandiset holds 4 control mice with 587 units and 6 experimental mice
with 554 units, against 5 with 585 and 7 with 546 in the STAR Methods. One mouse of each group is
missing, and each group nonetheless has more units than the paper reports (2 and 8 more).

## Claim figs4a-unilateral-no-behavior-deficit

> Mice with unilateral editing and imaging (the CRISPRoscopy mice) show no deficit in sniffing or
> aggression compared with controls.

Per mouse, from the annotations, over the interaction (from the first annotated behaviour to the end
of the file; the intruder's entry is not marked). Mice 510 and 500 have no attack annotation, so
attack measures use 3 control and 5 experimental mice. The interval between attacks is the mean gap
from the end of one bout to the start of the next (onset-to-onset shown too). The "attack
probability" of panel S4A(vii) is plotted against time (0 to 10 s), and the S3C legend lists "the
number and the time-varying probability" of attack bouts as two measures, with a decay constant ("tau
attack probability") in S2C and S3C. No summary per mouse is defined, so four readings are run:
(a) the probability of attacking at each time from 0 to 10 s after an attack onset, averaged over
onsets, and its mean over the 10 s; (b) the time at which that curve falls below 1/e; (c) the share
of the interaction spent attacking; (d) attack bouts per minute of interaction (the verifier's
reading; the S3C legend's "number" of bouts, divided by time). Time is given both in seconds and as a percentage of the interaction (the panels'
axes say "% of total time").

In [ ]:
rows = []
for m in mice:
    t0 = vmh.first_behaviour_s(m)
    T = len(m["Y"]) / m["rate"] - t0
    sn = m["bouts"]["sniff"]
    d = sn[:, 1] - sn[:, 0]
    r = {"mouse": m["mouse"], "group": m["group"], "interaction_s": T, "sniff_total_s": d.sum(),
         "sniff_pct": 100 * d.sum() / T, "sniff_bout_s": d.mean()}
    if "attack" in m["bouts"]:
        a = m["bouts"]["attack"]
        da = a[:, 1] - a[:, 0]
        r.update(attack_total_s=da.sum(), attack_pct=100 * da.sum() / T, attack_bout_s=da.mean(),
                 attack_bout_pct=100 * da.mean() / T, latency_s=a[0, 0] - t0,
                 interval_s=np.mean(a[1:, 0] - a[:-1, 1]), interval_onset_s=np.mean(np.diff(a[:, 0])),
                 attack_bouts=len(a), attack_per_min=60 * len(a) / T)
        att = vmh.frames(a, len(m["Y"]))
        curve = []
        for on, _ in a:                       # attack-triggered probability of attacking, 0 to 10 s
            i = int(round(on * m["rate"]))
            seg = np.full(101, np.nan)
            seg[:min(101, len(att) - i)] = att[i:i + 101]
            curve.append(seg)
        curve = np.nanmean(curve, axis=0)
        below = np.nonzero(curve < np.exp(-1))[0]
        r.update(attack_prob_0_10s=curve.mean(), attack_prob_tau_s=below[0] / m["rate"] if len(below) else np.nan)
    rows.append(r)
beh = pd.DataFrame(rows)
print(beh.round(2).to_string(index=False))
tests = []
for c in beh.columns[3:]:
    ctl, exp = beh[beh.group == "control"][c].dropna(), beh[beh.group == "experimental"][c].dropna()
    U, p = vmh.mw(ctl, exp)
    tests.append({"measure": c, "n_control": len(ctl), "n_exp": len(exp), "mean_control": ctl.mean(),
                  "mean_exp": exp.mean(), "exp/control": exp.mean() / ctl.mean(), "U": U, "p": p})
tests = pd.DataFrame(tests)
print("\nMann-Whitney, control against experimental:")
print(tests.round(3).to_string(index=False))
listed = ["sniff_total_s", "sniff_pct", "sniff_bout_s", "attack_total_s", "attack_pct", "attack_bout_s",
          "attack_bout_pct", "latency_s", "interval_s", "interval_onset_s"]
prob = {"(a) mean probability 0-10 s after an attack onset": "attack_prob_0_10s",
        "(b) decay to 1/e of that probability": "attack_prob_tau_s",
        "(c) share of the interaction attacking": "attack_pct", "(d) attack bouts per minute": "attack_per_min"}
print("\nattack probability, by reading: p =", {k: round(float(tests.set_index("measure").p[v]), 3) for k, v in prob.items()})
print("smallest p over the six other measures (seconds and percentages):", round(tests[tests.measure.isin(listed)].p.min(), 3))
print("\nsmallest p over the measures:", round(tests.p.min(), 3),
      "; smallest attainable two-sided p with 3 and 5 mice:", round(2 / 56, 3))

In [ ]:
# The paper's own reading of "no deficits" (see the conclusion): panels S4A(iii) and (vi) carry a star,
# with the experimental mice attacking more, so "no deficit" may mean no reduction. One-sided test of a
# deficit in experimental mice: less sniffing or attacking, shorter bouts, longer latency and interval,
# lower attack probability under readings (a) to (d).
worse = {"sniff_total_s": "less", "sniff_pct": "less", "sniff_bout_s": "less", "attack_total_s": "less",
         "attack_pct": "less", "attack_bout_s": "less", "attack_bout_pct": "less", "latency_s": "greater",
         "interval_s": "greater", "interval_onset_s": "greater", "attack_bouts": "less", "attack_per_min": "less",
         "attack_prob_0_10s": "less", "attack_prob_tau_s": "less"}
one = []
for c, alt in worse.items():
    ctl, exp = beh[beh.group == "control"][c].dropna(), beh[beh.group == "experimental"][c].dropna()
    one.append({"measure": c, "deficit_if_experimental_is": alt, "p_one_sided": vmh.mw(exp, ctl, alternative=alt)[1]})
one = pd.DataFrame(one)
print(one.round(3).to_string(index=False))
print("smallest one-sided p for a deficit:", round(one.p_one_sided.min(), 3))

**Result: reproduced, resting on two assumptions: the start of the interaction (for the latency
and the percentages) and the reading of "attack probability".** None of the six measures with a
defined summary differs between groups (smallest p = 0.114, sniff time as a percentage of the
interaction; the attainable minimum for the attack measures is 0.036). Attack probability does not
differ under readings (a), the mean attack-triggered probability over 0 to 10 s (0.45 against 0.36,
p = 0.25), (b), its decay to 1/e (4.7 against 3.0 s, p = 0.25), or (c), the share of time attacking
(4.3 against 10.7%, p = 0.14). Under reading (d), attack bouts per minute, the groups differ (0.80
against 2.79, p = 0.036, the smallest p 3 and 5 mice allow), with the experimental mice attacking
more often. Reading (a) is adopted: panel S4A(vii) plots the probability against time from 0 to 10
s, and the S3C legend lists "the number and the time-varying probability" of attack bouts as two
measures, so a bout rate is the number of bouts per time, not the probability. Under reading (d)
the claim's criterion would fail on a difference towards more aggression, not a deficit.

The claims file may misread the paper here, and a reader is asked (`questions/001131.yaml`). In
Figure S4A two panels carry a star: (iii), % time attacking, and (vi), interval between attacks,
both with the experimental mice more aggressive. So the paper's "no deficits" means no reduction,
and the claims file's "none differs at p < 0.05" is stricter than the paper. Tested as the
paper's reading (a variant), no measure shows a deficit in experimental mice on a one-sided test
(smallest one-sided p = 0.086, sniff bout duration), under every reading of attack probability,
including (d). Here the two starred measures do not differ (p = 0.14 and 0.39), with 3 and 5 mice.

## Claim fig4ef-male-activity-slightly-lower

> During interactions with a male intruder, the activity of all VMHvl Esr1 units (sigma above
> pre-intruder baseline, and z-scored) is slightly but significantly lower in experimental than in
> control mice.

Per unit, mean activity over the first 1 or 2 min from the first annotated behaviour, in units of
sigma (baseline readings `all` and `7s`, variant `gap10`) and z-scored. Tests under the two readings
of "nested": pooled units (`p_mw_pooled`, `p_ks_pooled`) and per-mouse means or medians
(`p_mw_mice`, `p_mw_mouse_medians`).

In [ ]:
def activity_table(select=None):
    rows = []
    for minutes in (1, 2):
        for base in ("all", "7s", "gap10"):
            vs, vz = {}, {}
            for m in mice:
                s = vmh.sigma_trace(m, base)
                if s is None:
                    continue
                w = vmh.window_frames(m, minutes)
                v = s[w].mean(0)
                keep = np.ones(len(v), bool) if select is None else select(v)
                vs[m["mouse"]] = v[keep]
                vz[m["mouse"]] = m["Y"][w].mean(0)[keep]
            a, b = vmh.nested_tests(vs, G), vmh.nested_tests(vz, G)
            rows.append({"minutes": minutes, "baseline": base, **a,
                         "z_mean_c": b["pooled_mean_c"], "z_mean_e": b["pooled_mean_e"],
                         "z_p_mw_pooled": b["p_mw_pooled"], "z_p_mw_mice": b["p_mw_mice"],
                         "z_p_ks_pooled": b["p_ks_pooled"], "z_p_mw_mouse_medians": b["p_mw_mouse_medians"]})
    return pd.DataFrame(rows)

act = activity_table()
print(act.round(4).to_string(index=False))
print("\nper-mouse mean sigma (baseline all, 1 min):")
per = {m["mouse"]: vmh.sigma_trace(m, "all")[vmh.window_frames(m, 1)].mean(0).mean()
       for m in mice if vmh.sigma_trace(m, "all") is not None}
print({k: round(v, 2) for k, v in per.items()})
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for i, minutes in enumerate((1, 2)):
    for g, col in (("control", "k"), ("experimental", "r")):
        v = np.concatenate([vmh.sigma_trace(m, "all")[vmh.window_frames(m, minutes)].mean(0)
                            for m in mice if m["group"] == g and vmh.sigma_trace(m, "all") is not None])
        ax[i].plot(np.sort(v), np.linspace(0, 1, len(v)), col, label=f"{g} ({len(v)} units)")
        print(f"{minutes} min {g}: sigma range {v.min():.2f} to {v.max():.2f}")
    ax[i].set(xlabel="mean activity (sigma, baseline all)", ylabel="ECDF", title=f"first {minutes} min")
    ax[i].legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Sensitivity: the same table without mice 497, 510 and 500, whose clock alignment could not be
# established (data check above).
unaligned = {"mutm497", "ctrlm510", "mutm500"}
keep_mice = mice
mice = [m for m in keep_mice if m["mouse"] not in unaligned]
act_aligned = activity_table()
mice = keep_mice
print("without", sorted(unaligned))
print(act_aligned[["minutes", "baseline", "n_mice_c", "n_mice_e", "n_units_c", "n_units_e", "pooled_mean_c", "pooled_mean_e",
                   "p_mw_pooled", "p_ks_pooled", "p_mw_mice", "p_mw_mouse_medians"]].round(4).to_string(index=False))
print("\nall mice: largest pooled Mann-Whitney p %.4f, largest pooled KS p %.4f" % (act.p_mw_pooled.max(), act.p_ks_pooled.max()))

**Result: reproduced, under the pooled reading of "nested" and resting on the assumed baseline and
start of interaction.** With units pooled, experimental activity in sigma is lower than control by
both the Mann-Whitney and the KS test at p < 0.05 under every baseline reading and both windows
(largest pooled p 0.014 and 0.0012; for example 0.53 against 0.95 sigma, p = 0.002 and KS p < 0.001,
baseline `all`, 1 min). The claims file records the paper's level as p < 0.01: that holds under the
`all` and `gap10` baselines (Mann-Whitney p 0.0002 to 0.002) but not for the 1-min window with the
`7s` baseline (p = 0.014). On per-mouse means or medians (4 control, 5 experimental mice) no
comparison reaches p < 0.05 (p from 0.06 to 0.73). Without mice 497, 510 and 500, whose clock
alignment could not be established, the pooled tests still give p < 0.01 under the `all` and `gap10`
baselines, but the Mann-Whitney test with the `7s` baseline does not reach p < 0.05 (p = 0.051 and
0.060). The difference is not slight: the experimental mean is about half the control mean. The
z-scored activity is also lower in experimental mice when pooled (KS p < 0.01 in both windows), but
its Mann-Whitney test does not reach p < 0.05.

## Claim figs4e-active-units-unchanged

> Among male-responsive units (at least 2 sigma above baseline), mean activity does not differ
> between control and experimental mice.

In [ ]:
active = activity_table(select=lambda v: v >= vmh.ACTIVE_SIGMA)
print(active[["minutes", "baseline", "n_mice_c", "n_mice_e", "n_units_c", "n_units_e", "pooled_mean_c",
              "pooled_mean_e", "mouse_mean_c", "mouse_mean_e", "p_mw_pooled", "p_mw_mice"]].round(4).to_string(index=False))

In [ ]:
# Sensitivity: without mice 497, 510 and 500, whose clock alignment could not be established.
mice = [m for m in keep_mice if m["mouse"] not in unaligned]
active_aligned = activity_table(select=lambda v: v >= vmh.ACTIVE_SIGMA)
mice = keep_mice
print("without", sorted(unaligned))
print(active_aligned[["minutes", "baseline", "n_mice_c", "n_mice_e", "n_units_c", "n_units_e", "pooled_mean_c",
                      "pooled_mean_e", "mouse_mean_c", "mouse_mean_e", "p_mw_pooled", "p_mw_mice"]].round(4).to_string(index=False))

**Result: reproduced, resting on an assumption: the baseline is all frames before the first
annotated behaviour.** With that baseline (and with the `gap10` variant), neither reading of
"nested" finds a difference: pooled p = 0.09 at 1 min and 0.76 at 2 min, per-mouse p = 0.56 and
0.90 (`gap10`: 0.06, 0.27, 0.29, 0.56). With the `7s` baseline the per-mouse test finds none (p =
0.19) but the pooled test finds control higher at p < 0.001 (12.8 against 8.5 sigma at 1 min), so
the outcome depends on the baseline under the pooled reading. The `all` baseline is adopted: the
Methods' baseline is a 5-min period alone in the home cage (Behavior annotations), of which the
frames before the first annotated behaviour are the nearest stand-in, while the last 7 s is only the
Methods' minimum ("during 7sec or more") and gives a narrow baseline that inflates sigma (1.8 times as many control units, 157 against 279 at 1 min, and 2.5 times as many experimental
units, 76 against 190, pass 2 sigma). The same `all` baseline and pooled reading reproduce
fig4ef-male-activity-slightly-lower.

Without mice 497, 510 and 500, whose clock alignment could not be established, the result is not
robust: with the `all` baseline the pooled test finds control higher at 1 min (3.67 against 3.03
sigma, p = 0.012, 110 control and 17 experimental units) but not at 2 min (p = 0.63); per mouse
(3 and 3 mice) p = 0.2 and 1.0. The outcome is the one on all mice in the dandiset.

## Claim fig5bd-behavior-selective-units-reduced

> During male interactions, sniff-selective units are reduced from 8.9 +/- 3% to 3.9 +/- 2% and
> attack-selective units from 6 +/- 0.9% to 1.8 +/- 0.7% in experimental mice; the mixed-selectivity
> fraction is reduced by about 22%, and units active during male-directed behaviors fall from 36.2%
> to 22.4%; female-directed selective fractions are similar.

Mice with both sniffing and attack annotations and a baseline: control 482, 508, 511; experimental
440, 441, 497, 501. Readings (a) and (b) of "active during all male-directed behaviors" give the same
number (sniff-selective + attack-selective + mixed), so one is shown.

In [ ]:
expected = {"sniff_sel_e": (3.9, 2), "sniff_sel_c": (8.9, 3), "att_sel_e": (1.8, 0.7), "att_sel_c": (6, 0.9),
            "active_c": (36.2, 3.62), "active_e": (22.4, 2.24), "mixed_reduction": (22, 2.2)}
sel_rows, per_mouse = [], []
for base in ("all", "7s", "gap10"):
    per = []
    for m in mice:
        if "attack" not in m["bouts"]:
            continue
        s = vmh.sigma_trace(m, base)
        if s is None:
            continue
        n = len(s)
        sn = s[vmh.frames(m["bouts"]["sniff"], n)].mean(0)
        at = s[vmh.frames(m["bouts"]["attack"], n)].mean(0)
        per.append({"baseline": base, "mouse": m["mouse"], "group": m["group"], "units": len(sn),
                    "sniff_sel": int(((sn >= 2) & (at < 2)).sum()), "att_sel": int(((at >= 2) & (sn < 2)).sum()),
                    "mixed": int(((sn >= 2) & (at >= 2)).sum())})
    per = pd.DataFrame(per)
    per["active"] = per.sniff_sel + per.att_sel + per.mixed
    per_mouse.append(per)
    for how in ("pooled", "mouse_mean"):
        r = {"baseline": base, "fractions": how}
        for g, d in per.groupby("group"):
            for c in ("sniff_sel", "att_sel", "mixed", "active"):
                r[f"{c}_{g[0]}"] = (100 * d[c].sum() / d.units.sum() if how == "pooled"
                                    else (100 * d[c] / d.units).mean())
        r["mixed_reduction"] = 100 * (1 - r["mixed_e"] / r["mixed_c"])
        r["within"] = sum(abs(r[k] - v) <= t for k, (v, t) in expected.items())
        sel_rows.append(r)
per_mouse = pd.concat(per_mouse)
print(per_mouse.to_string(index=False))
sel = pd.DataFrame(sel_rows)
print("\nPercentages (c control, e experimental); 'within' counts the 7 expected values inside tolerance:")
print(sel.round(1).to_string(index=False))
for _, r in sel.iterrows():
    print(r.baseline, r.fractions, "outside tolerance:",
          [k for k, (v, t) in expected.items() if abs(r[k] - v) > t])
for base in ("all", "7s"):
    d = per_mouse[per_mouse.baseline == base]
    for c in ("sniff_sel", "att_sel", "active"):
        f = 100 * d[c] / d.units
        print(base, c, "Mann-Whitney across mice p = %.3f" % vmh.mw(f[d.group == "control"], f[d.group == "experimental"])[1])

In [ ]:
# Sensitivity: without mouse 497, whose clock alignment could not be established (mice 510 and 500
# have no attack annotation and are not in this measure).
rows = []
for base in ("all", "7s", "gap10"):
    d = per_mouse[(per_mouse.baseline == base) & ~per_mouse.mouse.isin(unaligned)]
    r = {"baseline": base}
    for g, dg in d.groupby("group"):
        for c in ("sniff_sel", "att_sel", "mixed", "active"):
            r[f"{c}_{g[0]}"] = 100 * dg[c].sum() / dg.units.sum()
    r["mixed_reduction"] = 100 * (1 - r["mixed_e"] / r["mixed_c"])
    r["within"] = sum(abs(r[k] - v) <= t for k, (v, t) in expected.items())
    rows.append(r)
print("without mutm497, pooled fractions (%):")
print(pd.DataFrame(rows).round(1).to_string(index=False))

**Result: partially reproduced, resting on the assumed baseline.** The directions hold under every
reading: sniff-selective, attack-selective and active fractions are all lower in experimental mice
(table). The values do not all fall inside tolerance under any one reading. With the `all` baseline
and pooled fractions, control sniff-selective 9.6% and attack-selective 6.0% match the paper, and
experimental attack-selective 2.4% is inside 1.8 +/- 0.7%, but experimental sniff-selective (1.7%)
is below 3.9 +/- 2%, the active fractions (28.8% and 11.4%) are below 36.2% and 22.4%, and the
mixed fraction falls by 45%, not 22%. With the `7s` baseline more units pass 2 sigma and the control
values exceed the paper's. Without mouse 497, whose clock alignment could not be established (510 and 500 have no attack
annotation and are not in this measure), the pooled fractions keep every direction and 3 values
inside tolerance with the `all` and `gap10` baselines (2 with `7s`). The female-directed fractions
are not in the dandiset.

## Claim figs5e-behavior-decoding-intact

> Linear decoders distinguish attack from sniffing equally well in control and experimental mice.

Per mouse, frame-wise linear SVM on balanced attack and sniff frames, stratified five-fold
cross-validation, 20 repeats (STAR Methods; claims file). Mice with attack annotations: 3 control,
5 experimental. Shuffled decoders reassign the bouts' labels at random (20 repeats here, against the
paper's 1,000, a standard choice to keep the run short; the shuffle does not decide the outcome).
The bout-wise split is a variant that keeps each bout in one fold. As the control mice have more
units (109 to 225) than the experimental ones (31 to 127), a variant draws 31 units at random in
each repeat (the smallest count among these mice), with its own shuffled decoder, at each C.

In [ ]:
rng = np.random.default_rng(5)
rows = []
for m in mice:
    if "attack" not in m["bouts"]:
        continue
    r = {"mouse": m["mouse"], "group": m["group"]}
    for C in (0.1, 1.0, 10.0):
        r[f"acc_C{C:g}"] = vmh.decode_attack_vs_sniff(m, rng, C=C)
        r[f"shuffled_C{C:g}"] = vmh.decode_attack_vs_sniff(m, rng, C=C, shuffle=True)
        r[f"units31_C{C:g}"] = vmh.decode_attack_vs_sniff(m, rng, C=C, n_units=31)
        r[f"units31_shuffled_C{C:g}"] = vmh.decode_attack_vs_sniff(m, rng, C=C, shuffle=True, n_units=31)
    r["by_bout_C1"] = vmh.decode_attack_vs_sniff(m, rng, C=1.0, by_bout=True)
    r["by_bout_shuffled_C1"] = vmh.decode_attack_vs_sniff(m, rng, C=1.0, by_bout=True, shuffle=True)
    rows.append(r)
dec = pd.DataFrame(rows)
print(dec.round(3).to_string(index=False))
print()
for c in dec.columns[2:]:
    a, b = dec[dec.group == "control"][c], dec[dec.group == "experimental"][c]
    print(f"{c:22s} control {a.mean():.3f}  experimental {b.mean():.3f}  difference {a.mean() - b.mean():+.3f}"
          f"  Mann-Whitney p = {vmh.mw(a, b)[1]:.3f}")
for g in ("control", "experimental"):
    d = dec[dec.group == g]
    for real, shuf in [(f"acc_C{C:g}", f"shuffled_C{C:g}") for C in (0.1, 1.0, 10.0)] + [
            (f"units31_C{C:g}", f"units31_shuffled_C{C:g}") for C in (0.1, 1.0, 10.0)]:
        print(f"{g}: {real} against {shuf}, Mann-Whitney across mice p = {vmh.mw(d[real], d[shuf])[1]:.3f}, "
              f"mean difference {(d[real] - d[shuf]).mean():+.3f}")
print("smallest attainable two-sided p with 3 and 5 mice:", round(2 / 56, 3))

In [ ]:
# Sensitivity: without mouse 497, whose clock alignment could not be established (mice 510 and 500
# have no attack annotation and are not in this test).
d = dec[~dec.mouse.isin(unaligned)]
print("without mutm497:", len(d[d.group == "control"]), "control and", len(d[d.group == "experimental"]), "experimental mice")
for c in dec.columns[2:]:
    a, b = d[d.group == "control"][c], d[d.group == "experimental"][c]
    print(f"{c:22s} control {a.mean():.3f}  experimental {b.mean():.3f}  Mann-Whitney p = {vmh.mw(a, b)[1]:.3f}")
print("smallest attainable two-sided p with 3 and 4 mice:", round(2 / 35, 3))

**Result: not reproduced.** With the paper's frame-wise split, every control mouse decodes better
than every experimental mouse at all three values of C, and the Mann-Whitney test across mice finds
the difference at p = 0.036, the smallest p 3 and 5 mice allow (table). The difference is small
(0.06 to 0.07 in accuracy). With 31 units drawn per mouse the difference is larger (0.12 to 0.13)
and still at p = 0.036 for every C, so the control mice's larger unit counts do not explain it. The
measure is weak: the frame-wise split leaves frames of the same bout in training and test sets, so
decoders trained on shuffled bout labels score almost as well (control 0.981 to 0.988, experimental
0.904 to 0.925 over C), and real and shuffled accuracy do not differ in either group at any C or unit
count (p from 0.10 to 1.0). The paper puts chance at 50%; here the shuffled decoders are far above
it. With whole bouts held out (a variant), accuracy is 0.79 in control and 0.53 in experimental mice
against shuffled 0.53 and 0.50, and the groups differ at p = 0.036.

The shuffled decoders also separate the groups (p = 0.036 with 31 units at every C; p = 0.036,
0.071 and 0.25 with all units), so the group difference exists without behaviour labels. The
outcome follows the criterion, but this measure does not show that decoding of behaviour is
impaired. Without mouse 497, whose clock alignment could not be established, every comparison
gives p = 0.057 or more, the smallest p that 3 and 4 mice allow, so no difference could reach p <
0.05; every control mouse still decodes better than every experimental mouse in the real decoders.

## Claim fig7c-achw-all-units

> Across all VMHvl Esr1 units, the mean ACHW during the first 1-2 minutes of male-male interaction is
> reduced in experimental mice (by about 50.9%), less than for first-dimension units.

ACHW per unit over the first 1 or 2 min from the first annotated behaviour. The autocorrelation
is unchanged by an affine map of the trace, so the claims file's z-scored and sigma readings give
the same ACHW, and the sigma reading needs no baseline. With one male intruder per file, the "mean
over intruder trials" reading is the same as the single autocorrelation. All 10 mice are used.
The whole recording is shown as a variant. The estimator of the
autocorrelation is not given (the paper cites Remedios et al. 2017 for the computation). Three are
run (`vmh.achw`): the window's mean removed and each lag's sum divided by n (the standard sample
autocorrelation, the first run's choice), the same divided by n - k, and no mean removed (the file's
mean, 0 for these traces) divided by n - k. A synthetic check first shows how each recovers a known
ACHW over windows of these lengths. The third estimator is computed on the z-scored trace only; on
the sigma trace it would differ, as it does not remove the window's mean.

In [ ]:
from scipy.signal import lfilter

# Synthetic check: AR(1) units with a known ACHW, 200 units per case.
srng = np.random.default_rng(7)
rows = []
for true_s in (5, 10, 20, 30):
    phi = 0.5 ** (1 / (true_s * vmh.RATE_HZ))       # autocorrelation phi**k falls to 0.5 at true_s
    for n in (600, 1200, 10000):
        X = lfilter([1.0], [1.0, -phi], srng.standard_normal((n + 3000, 200)), axis=0)[3000:]
        rows.append({"true_achw_s": true_s, "frames": n,
                     **{est: np.nanmean(vmh.achw(X, estimator=est)) for est in vmh.ACHW_ESTIMATORS}})
print("Mean estimated ACHW (s) of AR(1) units with a known ACHW (true_achw_s), by window length in frames (600 = 1 min):")
print(pd.DataFrame(rows).round(1).to_string(index=False))

rows = []
achw_values = {}
windows = {"1 min": lambda m: vmh.window_frames(m, 1), "2 min": lambda m: vmh.window_frames(m, 2),
           "whole file (variant)": lambda m: np.arange(len(m["Y"]))}
for est in vmh.ACHW_ESTIMATORS:
    for label, sel_frames in windows.items():
        v = {m["mouse"]: vmh.achw(m["Y"][sel_frames(m)], estimator=est) for m in mice}
        achw_values[(est, label)] = v
        r = vmh.nested_tests(v, G)
        rows.append({"estimator": est, "window": label, **r,
                     "reduction_pooled_pct": 100 * (1 - r["pooled_mean_e"] / r["pooled_mean_c"]),
                     "reduction_mice_pct": 100 * (1 - r["mouse_mean_e"] / r["mouse_mean_c"]),
                     "undefined": int(sum(np.isnan(x).sum() for x in v.values()))})
ach = pd.DataFrame(rows)
print(ach.round(4).to_string(index=False))
for est in vmh.ACHW_ESTIMATORS:
    print(f"\nper-mouse mean ACHW (s), 2 min, {est}:", {k: round(np.nanmean(x), 2) for k, x in achw_values[(est, "2 min")].items()})
print("expected reduction 50.9 +/- 5.09 %")
fig, ax = plt.subplots(figsize=(4.5, 3.2))
for g, col in (("control", "k"), ("experimental", "r")):
    v2 = achw_values[("window_mean_biased", "2 min")]
    x = np.concatenate([v2[k] for k in v2 if G[k] == g])
    ax.plot(np.sort(x), np.linspace(0, 1, len(x)), col, label=g)
    print(f"{g}: ACHW range (2 min) {np.nanmin(x):.1f} to {np.nanmax(x):.1f} s")
ax.set(xlabel="ACHW (s), first 2 min, window-mean biased estimator", ylabel="ECDF"); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Units whose autocorrelation never falls to half within the window (no-mean estimator) have an ACHW
# longer than the window. Two ways to keep them: (i) set them to the window's length, a lower bound,
# which still understates the mean; (ii) compare medians with them counted as the longest, which the
# censoring does not bias. The claim's measure is the mean, so (ii) is shown only for robustness.
rows = []
for minutes in (1, 2):
    v = achw_values[("file_mean_unbiased", f"{minutes} min")]
    cap = {k: np.where(np.isnan(x), len(vmh.window_frames(M[k], minutes)) / vmh.RATE_HZ, x) for k, x in v.items()}
    r = vmh.nested_tests(cap, G)
    inf = {g: np.concatenate([np.where(np.isnan(v[k]), np.inf, v[k]) for k in v if G[k] == g])
           for g in ("control", "experimental")}
    rows.append({"window": f"{minutes} min",
                 "undefined_c": int(sum(np.isnan(v[k]).sum() for k in v if G[k] == "control")),
                 "undefined_e": int(sum(np.isnan(v[k]).sum() for k in v if G[k] == "experimental")),
                 "capped_mean_c": r["pooled_mean_c"], "capped_mean_e": r["pooled_mean_e"],
                 "capped_reduction_pooled_pct": 100 * (1 - r["pooled_mean_e"] / r["pooled_mean_c"]),
                 "capped_reduction_mice_pct": 100 * (1 - r["mouse_mean_e"] / r["mouse_mean_c"]),
                 "p_mw_pooled": r["p_mw_pooled"], "p_mw_mice": r["p_mw_mice"],
                 "median_c": np.median(inf["control"]), "median_e": np.median(inf["experimental"]),
                 "median_reduction_pct": 100 * (1 - np.median(inf["experimental"]) / np.median(inf["control"]))})
print("No-mean estimator, undefined units kept:")
print(pd.DataFrame(rows).round(4).to_string(index=False))

# Sensitivity: without mice 497, 510 and 500, whose clock alignment could not be established.
rows = []
for (est, label), v in achw_values.items():
    v = {k: x for k, x in v.items() if k not in unaligned}
    r = vmh.nested_tests(v, G)
    rows.append({"estimator": est, "window": label, "n_mice_c": r["n_mice_c"], "n_mice_e": r["n_mice_e"],
                 "reduction_pooled_pct": 100 * (1 - r["pooled_mean_e"] / r["pooled_mean_c"]),
                 "reduction_mice_pct": 100 * (1 - r["mouse_mean_e"] / r["mouse_mean_c"]),
                 "p_mw_pooled": r["p_mw_pooled"], "p_ks_pooled": r["p_ks_pooled"], "p_mw_mice": r["p_mw_mice"]})
print("\nwithout", sorted(unaligned))
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: not reproduced.** Over the first 1 or 2 min the reduction is 15 to 31% under every
estimator and both means (window mean removed, divided by n: 14.6 and 15.9% pooled, 19.2% per mouse;
divided by n - k: 15.5 to 20.2%; no mean removed, divided by n - k: 24.8 to 31.5%), not 50.9 +/- 5.1%.
Experimental units are shorter with units pooled (Mann-Whitney and KS p < 0.001 under every
estimator) but not across mice (p from 0.07 to 0.26). The synthetic check shows that the window-mean
estimators shorten a known ACHW of 10 to 30 s to 4 to 11 s over 1 to 2 min, which compresses group
differences; the no-mean estimator comes within 30% of the truth at 2 min (5.2, 12.8, 23.2 and 29.1
s for 5, 10, 20 and 30 s) but leaves 46 to 81 units undefined (their autocorrelation stays above half within the window). Over the whole recording
(a variant, not the claim's window) the reduction is 43 to 47%, inside or near the tolerance.

The result is not robust to the estimator. The undefined units of the no-mean estimator are the
slowest, and more of them are control units (50 against 31 at 1 min, 34 against 12 at 2 min), so
leaving them out understates the reduction. Set to the window's length, a lower bound, they give a
reduction of mean ACHW of 27.6 and 34.5% pooled (24.8 and 31.7% over mice), still outside
tolerance. The median, which this censoring does not bias, falls by 49.6 and 53.3%, inside the
tolerance; the claim's measure is the mean, so the outcome stands. Without mice 497, 510 and 500,
whose clock alignment could not be established, the reduction over 1 to 2 min is 5 to 18%.

## Claim figs5c-peth-activity

> Mean bout-triggered activity is slightly but significantly lower in experimental mice during
> male-directed sniffing and attack, higher during female-directed sniffing, and unchanged during
> mounting.

Per unit and bout, the z-scored trace over 15 s from onset minus its mean over a 2-s or 3-s pre-bout
baseline, averaged over bouts; nested Mann-Whitney under both readings. The criterion needs all four
directions; the two female-directed ones cannot be tested here, but the male-directed ones can, and
if they fail the claim fails whatever the female data would show.

In [ ]:
rows = []
for pre in (2, 3):
    for b in ("sniff", "attack"):
        v = {}
        for m in mice:
            if b not in m["bouts"]:
                continue
            Y, n = m["Y"], len(m["Y"])
            per_bout = []
            for on, _ in m["bouts"][b]:
                i = int(round(on * 10))
                if i - pre * 10 < 0 or i + 150 > n:
                    continue
                per_bout.append(Y[i:i + 150].mean(0) - Y[i - pre * 10:i].mean(0))
            v[m["mouse"]] = np.mean(per_bout, axis=0)
        rows.append({"pre_bout_baseline_s": pre, "behaviour": b, **vmh.nested_tests(v, G)})
peth = pd.DataFrame(rows)
print(peth.round(4).to_string(index=False))
print("\nmale sniff and attack lower at p < 0.05 (both directions needed):",
      {f"{r.pre_bout_baseline_s} s, {r.behaviour}": {"pooled": bool(r.p_mw_pooled < 0.05 and r.pooled_mean_e < r.pooled_mean_c),
                                                      "mice": bool(r.p_mw_mice < 0.05 and r.mouse_mean_e < r.mouse_mean_c)}
       for r in peth.itertuples()})

**Result: not reproduced.** The male-directed half of the criterion fails: bout-triggered activity
is not lower in experimental mice at p < 0.05 by the nested Mann-Whitney test, for sniffing or
attack, with either pre-bout baseline, pooled or per mouse (p from 0.11 to 0.61 on pooled units and
per-mouse means, up to 0.79 on per-mouse medians). The experimental
means are lower (for example sniff 0.002 against 0.051, 2-s baseline, pooled), and the KS test,
which is not the claim's test, separates the distributions (p < 0.002). The female-directed half
(higher during female sniffing, unchanged during mounting) is not in the dandiset and is not
tested; it cannot rescue the claim, which needs all four directions.

## Claims blocked for missing data, with their male part

fig5ac-behavior-activity-male-lower, fig4g-male-selective-reduced and fig5e-cp-tuning-shift need
the female-intruder sessions, which the dandiset does not hold. Their male-directed parts are shown
here and do not decide an outcome.

In [ ]:
rows = []
for base in ("all", "7s"):
    for b in ("sniff", "attack"):
        vs = {}
        for m in mice:
            s = vmh.sigma_trace(m, base)
            if s is None or b not in m["bouts"]:
                continue
            vs[m["mouse"]] = s[vmh.frames(m["bouts"][b], len(s))].mean(0)
        rows.append({"baseline": base, "behaviour": b, **vmh.nested_tests(vs, G)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

### fig4g-male-selective-reduced, the male-activated fractions

A unit is male-activated when its mean activity over the male window is at least 2 sigma above
the baseline (selective or mixed; Figure 4G). Male-selective and mixed units need the female
interaction; the fraction of male-activated units does not, and is shown per mouse and pooled, for
each baseline and window, against 38.3 +/- 3.83% (control) and 34.6 +/- 3.46% (experimental).

In [ ]:
rows, frac = [], []
for minutes in (1, 2):
    for base in ("all", "7s", "gap10"):
        per = []
        for m in mice:
            s = vmh.sigma_trace(m, base)
            if s is None:
                continue
            v = s[vmh.window_frames(m, minutes)].mean(0)
            per.append({"minutes": minutes, "baseline": base, "mouse": m["mouse"], "group": m["group"],
                        "units": len(v), "male_activated": int((v >= vmh.ACTIVE_SIGMA).sum())})
        per = pd.DataFrame(per)
        per["pct"] = 100 * per.male_activated / per.units
        rows.append(per)
        r = {"minutes": minutes, "baseline": base}
        for g, d in per.groupby("group"):
            r[f"pooled_{g[:3]}"] = 100 * d.male_activated.sum() / d.units.sum()
            r[f"mouse_mean_{g[:3]}"] = d.pct.mean()
        r["p_mw_mice"] = vmh.mw(per[per.group == "control"].pct, per[per.group == "experimental"].pct)[1]
        r["control_within_38.3+/-3.83"] = any(abs(r[k] - 38.3) <= 3.83 for k in ("pooled_con", "mouse_mean_con"))
        r["exp_within_34.6+/-3.46"] = any(abs(r[k] - 34.6) <= 3.46 for k in ("pooled_exp", "mouse_mean_exp"))
        frac.append(r)
print(pd.concat(rows).pivot_table(index=["group", "mouse"], columns=["minutes", "baseline"], values="pct").round(1).to_string())
print("\nmale-activated units, % (pooled and mean over mice):")
print(pd.DataFrame(frac).round(3).to_string(index=False))

The male-activated fraction is lower in experimental mice under every baseline and window
(table; across mice p from 0.016 to 0.41). The control fraction is outside 38.3 +/- 3.83% under every
reading (23.3 to 47.5% pooled, 23.4 to 46.7% as a mean over mice); the experimental fraction is inside
34.6 +/- 3.46% only with the `7s` baseline, and there only as the mean over mice at 1 min (37.4%)
and both summaries at 2 min (37.1% pooled, 36.4% over mice); pooled at 1 min it is 40.1%, outside. The male-selective fractions need the female interaction.

### fig5e-cp-tuning-shift, the male attack-against-sniff part

Choice probability of each unit for attack against sniffing (`vmh.choice_probability`), with 100
bout shuffles under the two shuffle readings. CP is the area under the ROC curve, so CP(sniff vs
attack) = 1 - CP(attack vs sniff) and the two direction readings give the same units (tuned to
sniff: CP(attack vs sniff) < 0.3 and below the shuffled mean by 2 SD), and it is unchanged by an
increasing affine map, so the z-scored and sigma traces give the same CP (the sigma trace only drops
mouse 488, which has no baseline; shown as a separate set of mice). "Mixed" is taken literally from
the claims file, units with CP at most 0.7 both ways (tuned to neither). Expected: attack-tuned
reduced by 80 +/- 8%, sniff-tuned by 70 +/- 7%, mixed increased by 18 +/- 1.8%.

In [ ]:
rng = np.random.default_rng(11)
cp_rows = []
for m in mice:
    if "attack" not in m["bouts"]:
        continue
    for sh in ("circular", "reassign"):
        r = vmh.choice_probability(m, "attack", "sniff", rng, shuffle=sh)
        cp_rows.append({"mouse": m["mouse"], "group": m["group"], "shuffle": sh, "units": len(r["cp"]),
                        "attack_tuned": int(r["tuned_a"].sum()), "sniff_tuned": int(r["tuned_b"].sum()),
                        "cp_above_0.7": int((r["cp"] > 0.7).sum()), "cp_below_0.3": int((r["cp"] < 0.3).sum()),
                        "null_sd_median": float(np.median(r["null_sd"]))})
cpt = pd.DataFrame(cp_rows)
print(cpt.round(3).to_string(index=False))
out = []
for sh in ("circular", "reassign"):
    for mice_set, drop in (("all with attack", set()), ("with a baseline (sigma)", {"mutm488"})):
        d0 = cpt[(cpt["shuffle"] == sh) & ~cpt.mouse.isin(drop)]
        for how in ("pooled", "mouse_mean"):
            r = {"shuffle": sh, "mice": mice_set, "fractions": how}
            for g, d in d0.groupby("group"):
                for col in ("attack_tuned", "sniff_tuned"):
                    r[f"{col}_{g[:3]}"] = (100 * d[col].sum() / d.units.sum() if how == "pooled"
                                           else (100 * d[col] / d.units).mean())
                r[f"mixed_{g[:3]}"] = 100 - r[f"attack_tuned_{g[:3]}"] - r[f"sniff_tuned_{g[:3]}"]
            r["attack_reduction"] = 100 * (1 - r["attack_tuned_exp"] / r["attack_tuned_con"])
            r["sniff_reduction"] = 100 * (1 - r["sniff_tuned_exp"] / r["sniff_tuned_con"])
            r["mixed_increase"] = 100 * (r["mixed_exp"] / r["mixed_con"] - 1)
            r["within"] = int(abs(r["attack_reduction"] - 80) <= 8) + int(abs(r["sniff_reduction"] - 70) <= 7) + int(abs(r["mixed_increase"] - 18) <= 1.8)
            out.append(r)
print("\nfractions of units, %, and relative changes (within counts the three male values inside tolerance):")
print(pd.DataFrame(out).round(1).to_string(index=False))

Under every combination run, attack-tuned units are fewer in experimental mice (reduction 55.5
to 82.7%), sniff-tuned units are fewer (13.8 to 83.1%) and units tuned to neither are more (2.3 to
25.4%). At most two of the three male values are inside tolerance in any combination (reassigned
bouts, mean over mice: attack 78.9% and sniff 76.8% inside, mixed 24.0% outside 18 +/- 1.8%). The
circular shuffle has a wide null (median SD of the shuffled CP 0.06 to 0.30), so it finds fewer tuned
units. The two values for female sniffing cannot be computed.

## Claim fig1e-oxtr-avpr1a-coexpression

> About 78% of Esr1+ VMHvl cells express Oxtr mRNA, and about 61% of these also express Avpr1a
> mRNA, in a single-cell RNA-seq atlas of VMHvl.

Rounds 1 and 2 left this claim blocked (other), as the atlas had not been found; it was tested
after round 2, and round 3 agreed with the outcome. It is the atlas of Kim et al. 2019 (doi:10.1016/j.cell.2019.09.020, the paper's reference 67), deposited
at Mendeley Data (doi:10.17632/ypx3sw2f7c.1, found through the article's data links in Europe PMC).
Figure 1B plots counts per million reads (CPM) by cluster, which the atlas's SMART-seq part gives
(`vmh.load_atlas`). The STAR Methods say only that the data "were analyzed as described previously"
(Transcriptomic analysis), with no expression threshold and no subset of cells. The Esr1+ VMHvl
cells are read as the cells of the atlas's eight clusters named for Esr1 (`Esr1_1` to `Esr1_6`,
`Tsix_Esr1_1`, `Tsix_Esr1_2`), which matches the number in the Results' "most clusters (5/8)". Readings run:
the cell set (all cells of those clusters; male cells only, as the paper's mice are male; cells of
those clusters in which Esr1 itself is detected) and the threshold for "expressed" (CPM above 0, 1,
5 or 10).

In [ ]:
cpm, ameta = vmh.load_atlas()
clusters = ameta.smart_seq_cluster_label.astype(str)
esr1_cl = clusters.str.contains("Esr1")
print(f"{len(cpm)} SMART-seq cells; Esr1 clusters: {sorted(clusters[esr1_cl].unique())}")
print(ameta[esr1_cl].groupby(["smart_seq_cluster_label", "sex_label"]).size().unstack().to_string())
rows = []
for thr in (0, 1, 5, 10):
    for cells, sel in (("all cells of the Esr1 clusters", esr1_cl),
                       ("male cells only", esr1_cl & (ameta.sex_label == "M")),
                       ("Esr1 detected", esr1_cl & (cpm.Esr1 > thr))):
        o, a = cpm.Oxtr[sel] > thr, cpm.Avpr1a[sel] > thr
        rows.append({"cpm_above": thr, "cells": cells, "n": int(sel.sum()), "oxtr_pct": 100 * o.mean(),
                     "avpr1a_of_oxtr_pct": 100 * a[o].mean(), "both_of_esr1_pct": 100 * (o & a).mean()})
co = pd.DataFrame(rows)
co["oxtr_within"] = (co.oxtr_pct - 78).abs() <= 7.8
co["avpr1a_within"] = (co.avpr1a_of_oxtr_pct - 61).abs() <= 6.1
print("\nexpected 78 +/- 7.8% and 61 +/- 6.1%")
print(co.round(1).to_string(index=False))
per_cl = pd.DataFrame({"cluster": clusters[esr1_cl], "oxtr": cpm.Oxtr[esr1_cl] > 0,
                       "avpr1a": cpm.Avpr1a[esr1_cl] > 0})
per_cl["both"] = per_cl.oxtr & per_cl.avpr1a
print("\nper cluster, CPM above 0 (fraction of cells):")
print(per_cl.groupby("cluster")[["oxtr", "avpr1a", "both"]].mean().round(2).to_string())
print("\nAvpr1b or Avpr2 above 0 in the Esr1 clusters: %.1f%% and %.1f%%"
      % (100 * (cpm.Avpr1b[esr1_cl] > 0).mean(), 100 * (cpm.Avpr2[esr1_cl] > 0).mean()))

**Result: partially reproduced, resting on an assumption: all cells of the eight Esr1
clusters, with a gene expressed at any CPM above 0.** (Blocked until round 3, which agreed with this
outcome.) Under that reading 73.9% of the 850 Esr1-cluster cells express Oxtr, inside
78 +/- 7.8%, and 49.4% of these also express Avpr1a, below 61 +/- 6.1%. The readings give different
outcomes (table). Only male cells at CPM above 0 put both values inside tolerance (70.4% and 56.2%,
both near the lower edge); every reading with a threshold of 5 CPM or more puts both outside, and
the Avpr1a fraction is outside under every threshold, both for all cells of the clusters (43.6 to
49.4%) and for those with Esr1 detected (43.0 to 48.6%). The adopted
reading is the closest to the paper's words, "Esr1+ cells in VMHvl", which name no sex and no
threshold, and detection above 0 is the most inclusive threshold, the one that gives the highest
fractions. Cells with Esr1 detected give the same split (73.9% and 48.6%). At CPM above 0 all
eight clusters hold cells that co-express Avpr1a (the fewest, Tsix_Esr1_1, 3%), where the Results
say 5 of 8, so the "5/8" matches the number of Esr1 clusters but not the count of co-expressing
ones; the paper may have used a stricter criterion, though higher thresholds lower both fractions.
As the paper says, Avpr1b and Avpr2 are almost absent (0.0% and 0.2% of cells).

## Summary

| Claim | Outcome |
|---|---|
| methods-unit-counts | not reproduced: 4 and 6 mice, 587 and 554 units, against 5 and 7, 585 and 546 |
| figs4a-unilateral-no-behavior-deficit | reproduced (assumed: start of interaction; attack probability as an attack-triggered time course); attack bouts per minute differ, higher in experimental mice |
| fig4ef-male-activity-slightly-lower | reproduced with units pooled (assumed baseline); not across mice |
| figs4e-active-units-unchanged | reproduced (assumed: baseline of all frames before the first behaviour); the 7-s baseline with units pooled gives a difference |
| fig5bd-behavior-selective-units-reduced | partially reproduced (directions hold, values outside tolerance) |
| figs5e-behavior-decoding-intact | not reproduced: control decodes better, p = 0.036, also with 31 units per mouse; shuffled decoders score as well as real ones |
| fig7c-achw-all-units | not reproduced: reduction 15 to 31% over three estimators, not 50.9% |
| figs5c-peth-activity | not reproduced: male sniff and attack activity not lower at p < 0.05; the female half is not in the dandiset |
| fig5ac, fig4g-male, fig5e and the other female-intruder claims | blocked, data missing; male parts shown |
| fig1e-oxtr-avpr1a-coexpression | partially reproduced (assumed: all cells of the Esr1 clusters, CPM above 0): Oxtr 73.9%, Avpr1a among them 49.4%, not 61% |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. The dandiset gives z-scored traces only and marks neither
the pre-intruder baseline nor the intruder's entry, so the baseline is taken as the frames before the
first annotated behaviour, and the interaction as starting at that behaviour. One control and one
experimental mouse are not in the dandiset, and two mice have no attack annotation, so tests across
mice use 3 to 6 mice per group (4 and 6 where every mouse enters, 3 and 5 for attack measures). "Nested" tests are not defined in the paper; both readings are shown.
The estimator of the autocorrelation and the summary of "attack probability" are not given; every
reading run is shown. Mice 497, 510 and 500, whose clock alignment could not be established, are
left out in a sensitivity check of each sigma, decoding and ACHW claim.

What was found about the dataset. One mouse per group is missing; units per group differ from the
Methods' counts; the files of mice 510 and 500 describe z-scores as "DF/F" from "GCaMP6m" (the paper
names GCaMP8s) and hold only sniffing annotations; the baseline, the intruder's entry and removal and
the female-intruder sessions are absent; session start dates are placeholders; the earlier published
version's two files duplicate mice 510 and 500.